# 🛠️ Notebook 2: ATM — Implementation

We'll build the ATM from Notebook 1 end-to-end. The code is intentionally small and commented so you can follow every line.

**Features implemented**
- Card + PIN authentication with a **3-tries-then-block** rule.
- A proper **state machine** (`IDLE → CARD_INSERTED → AUTHENTICATED → TRANSACTING`).
- Four transactions as separate classes: `BalanceInquiry`, `Deposit`, `Withdraw`, `Transfer` (**Command pattern**).
- Checking **and** savings accounts.
- **Daily withdrawal limit** per account.
- A **cash dispenser** that runs out of money.
- A receipt **printer** and an internal **transaction log**.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/atm
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1) Domain: Bank, Customer, Account, Card

In [1]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import date
from enum import Enum
from typing import Dict, List, Optional


class AccountType(Enum):
    CHECKING = "checking"
    SAVINGS  = "savings"


@dataclass
class Account:
    id: str
    kind: AccountType
    balance: float = 0.0
    daily_withdraw_limit: float = 400.0
    # date -> amount already withdrawn that day
    _withdrawn_today: Dict[date, float] = field(default_factory=dict)

    def withdraw(self, amount: float, today: date) -> None:
        if amount <= 0:
            raise ValueError("amount must be positive")
        used = self._withdrawn_today.get(today, 0.0)
        if used + amount > self.daily_withdraw_limit:
            raise RuntimeError(
                f"daily limit ${self.daily_withdraw_limit:.0f} exceeded "
                f"(already took ${used:.0f})"
            )
        if amount > self.balance:
            raise RuntimeError("insufficient funds")
        self.balance -= amount
        self._withdrawn_today[today] = used + amount

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("amount must be positive")
        self.balance += amount


@dataclass
class Customer:
    name: str
    accounts: Dict[AccountType, Account]


@dataclass
class Card:
    number: str
    pin: str
    customer: Customer
    blocked: bool = False


class Bank:
    """The bank owns customers, cards and the rule 'block after 3 bad PINs'.

    In the real world this runs on bank servers and the ATM talks to it over
    the network. Here it's a simple in-memory object so the notebook runs
    without any external services.
    """

    MAX_PIN_TRIES = 3

    def __init__(self):
        self._cards: Dict[str, Card] = {}
        self._bad_tries: Dict[str, int] = {}

    def register(self, card: Card) -> None:
        self._cards[card.number] = card
        self._bad_tries[card.number] = 0

    def authenticate(self, card_number: str, pin: str) -> Card:
        card = self._cards[card_number]
        if card.blocked:
            raise RuntimeError("card blocked — contact your bank")
        if card.pin != pin:
            self._bad_tries[card_number] += 1
            if self._bad_tries[card_number] >= self.MAX_PIN_TRIES:
                card.blocked = True
                raise RuntimeError("too many wrong PINs — card blocked")
            raise RuntimeError(
                f"wrong PIN ({self._bad_tries[card_number]}/{self.MAX_PIN_TRIES})"
            )
        self._bad_tries[card_number] = 0  # reset on success
        return card


## 2) Hardware abstractions

One class per device. Makes the ATM easy to test with fakes.


In [2]:
class CashDispenser:
    def __init__(self, cash: float):
        self.cash = cash

    def dispense(self, amount: float) -> None:
        if amount > self.cash:
            raise RuntimeError("ATM cannot dispense — not enough cash in machine")
        self.cash -= amount


class DepositSlot:
    def __init__(self):
        self.collected = 0.0

    def accept(self, amount: float) -> None:
        self.collected += amount


class Screen:
    def show(self, msg: str) -> None:
        print("🖥️  ", msg)


class Printer:
    def print_receipt(self, lines: List[str]) -> None:
        print("🧾 RECEIPT")
        for line in lines:
            print("    " + line)
        print("   -----")


## 3) Transactions — the Command pattern

Each transaction is a small class that knows how to `execute` itself against the ATM. Adding a new one (say, `PayBill`) means adding a new class — no existing code changes.


In [3]:
class Transaction(ABC):
    @abstractmethod
    def execute(self, atm: "ATM") -> None: ...


class BalanceInquiry(Transaction):
    def __init__(self, account: Account):
        self.account = account

    def execute(self, atm):
        atm.screen.show(f"{self.account.kind.value} balance: ${self.account.balance:.2f}")
        atm.printer.print_receipt([
            f"Balance inquiry on {self.account.id}",
            f"Balance: ${self.account.balance:.2f}",
        ])
        atm.log.append(("balance", self.account.id, self.account.balance))


class Deposit(Transaction):
    def __init__(self, account: Account, amount: float):
        self.account = account
        self.amount = amount

    def execute(self, atm):
        atm.deposit_slot.accept(self.amount)
        self.account.deposit(self.amount)
        atm.screen.show(f"Deposited ${self.amount:.2f}")
        atm.printer.print_receipt([
            f"Deposit to {self.account.id}",
            f"Amount: ${self.amount:.2f}",
            f"New balance: ${self.account.balance:.2f}",
        ])
        atm.log.append(("deposit", self.account.id, self.amount))


class Withdraw(Transaction):
    def __init__(self, account: Account, amount: float):
        self.account = account
        self.amount = amount

    def execute(self, atm):
        # Check dispenser FIRST so we don't debit the account then fail.
        if self.amount > atm.dispenser.cash:
            raise RuntimeError("ATM cannot dispense — not enough cash in machine")
        self.account.withdraw(self.amount, atm.today)
        atm.dispenser.dispense(self.amount)
        atm.screen.show(f"Please take ${self.amount:.2f}")
        atm.printer.print_receipt([
            f"Withdraw from {self.account.id}",
            f"Amount: ${self.amount:.2f}",
            f"New balance: ${self.account.balance:.2f}",
        ])
        atm.log.append(("withdraw", self.account.id, self.amount))


class Transfer(Transaction):
    def __init__(self, source: Account, target: Account, amount: float):
        self.source, self.target, self.amount = source, target, amount

    def execute(self, atm):
        if self.amount <= 0:
            raise ValueError("amount must be positive")
        if self.amount > self.source.balance:
            raise RuntimeError("insufficient funds for transfer")
        # Not going through Account.withdraw() because daily-cash-limit
        # only applies to cash out of the machine, not internal transfers.
        self.source.balance -= self.amount
        self.target.deposit(self.amount)
        atm.screen.show(
            f"Transferred ${self.amount:.2f} "
            f"{self.source.kind.value} → {self.target.kind.value}"
        )
        atm.printer.print_receipt([
            f"Transfer {self.source.id} → {self.target.id}",
            f"Amount: ${self.amount:.2f}",
        ])
        atm.log.append(("transfer", self.source.id, self.target.id, self.amount))


## 4) The ATM controller with a state machine

In [4]:
class State(Enum):
    IDLE          = "idle"
    CARD_INSERTED = "card_inserted"
    AUTHENTICATED = "authenticated"


class ATM:
    def __init__(self, bank: Bank, dispenser: CashDispenser,
                 deposit_slot: DepositSlot, screen: Screen, printer: Printer,
                 today: Optional[date] = None):
        self.bank = bank
        self.dispenser = dispenser
        self.deposit_slot = deposit_slot
        self.screen = screen
        self.printer = printer
        self.today = today or date.today()

        self.state: State = State.IDLE
        self.card: Optional[Card] = None
        self.log: list = []

    # ---- state transitions ------------------------------------------------
    def insert_card(self, card: Card) -> None:
        if self.state != State.IDLE:
            raise RuntimeError("ATM is busy")
        if card.blocked:
            raise RuntimeError("card blocked — keeping card")
        self.card = card
        self.state = State.CARD_INSERTED
        self.screen.show("Card accepted. Please enter PIN.")

    def enter_pin(self, pin: str) -> None:
        if self.state != State.CARD_INSERTED:
            raise RuntimeError("insert card first")
        try:
            self.bank.authenticate(self.card.number, pin)
        except RuntimeError as err:
            self.screen.show(str(err))
            # If the bank just blocked the card, eject and reset.
            if self.card.blocked:
                self.eject()
            raise
        self.state = State.AUTHENTICATED
        self.screen.show(f"Welcome, {self.card.customer.name}!")

    def run(self, transaction: Transaction) -> None:
        if self.state != State.AUTHENTICATED:
            raise RuntimeError("not authenticated")
        transaction.execute(self)

    def eject(self) -> None:
        self.card = None
        self.state = State.IDLE
        self.screen.show("Card ejected. Have a nice day!")


## 5) Happy-path demo

In [5]:
from datetime import date

# --- set up the bank -------------------------------------------------------
alice = Customer(
    name="Alice",
    accounts={
        AccountType.CHECKING: Account("A-CHK", AccountType.CHECKING, balance=500.0),
        AccountType.SAVINGS:  Account("A-SAV", AccountType.SAVINGS,  balance=1500.0),
    },
)
card = Card(number="CARD-1", pin="1234", customer=alice)

bank = Bank()
bank.register(card)

# --- set up the ATM --------------------------------------------------------
atm = ATM(
    bank=bank,
    dispenser=CashDispenser(cash=1000.0),
    deposit_slot=DepositSlot(),
    screen=Screen(),
    printer=Printer(),
    today=date(2026, 4, 20),
)

# --- run a session ---------------------------------------------------------
atm.insert_card(card)
atm.enter_pin("1234")

chk = alice.accounts[AccountType.CHECKING]
sav = alice.accounts[AccountType.SAVINGS]

atm.run(BalanceInquiry(chk))
atm.run(Withdraw(chk, 200))           # cash out
atm.run(Deposit(sav, 100))            # deposit cash into savings
atm.run(Transfer(sav, chk, 300))      # internal transfer
atm.run(BalanceInquiry(chk))
atm.run(BalanceInquiry(sav))

atm.eject()
print("\nATM cash left:", atm.dispenser.cash)
print("Deposits collected:", atm.deposit_slot.collected)
print("Transaction log:", atm.log)


🖥️   Card accepted. Please enter PIN.
🖥️   Welcome, Alice!
🖥️   checking balance: $500.00
🧾 RECEIPT
    Balance inquiry on A-CHK
    Balance: $500.00
   -----
🖥️   Please take $200.00
🧾 RECEIPT
    Withdraw from A-CHK
    Amount: $200.00
    New balance: $300.00
   -----
🖥️   Deposited $100.00
🧾 RECEIPT
    Deposit to A-SAV
    Amount: $100.00
    New balance: $1600.00
   -----
🖥️   Transferred $300.00 savings → checking
🧾 RECEIPT
    Transfer A-SAV → A-CHK
    Amount: $300.00
   -----
🖥️   checking balance: $600.00
🧾 RECEIPT
    Balance inquiry on A-CHK
    Balance: $600.00
   -----
🖥️   savings balance: $1300.00
🧾 RECEIPT
    Balance inquiry on A-SAV
    Balance: $1300.00
   -----
🖥️   Card ejected. Have a nice day!

ATM cash left: 800.0
Deposits collected: 100.0
Transaction log: [('balance', 'A-CHK', 500.0), ('withdraw', 'A-CHK', 200), ('deposit', 'A-SAV', 100), ('transfer', 'A-SAV', 'A-CHK', 300), ('balance', 'A-CHK', 600.0), ('balance', 'A-SAV', 1300.0)]


## 6) Error paths — the interesting bits

These are the edge cases a real ATM has to handle. We **prove** each rule by triggering it.


In [6]:
# -- wrong PIN three times blocks the card ---------------------------------
bob = Customer("Bob", {AccountType.CHECKING: Account("B-CHK", AccountType.CHECKING, 200)})
bob_card = Card("CARD-2", "4321", bob)
bank.register(bob_card)

atm.insert_card(bob_card)
for attempt in range(3):
    try:
        atm.enter_pin("0000")
    except RuntimeError as e:
        print(f"attempt {attempt+1}: {e}")

print("card blocked?", bob_card.blocked)
print("ATM state after block:", atm.state)

# -- trying to use a blocked card is refused up front ----------------------
try:
    atm.insert_card(bob_card)
except RuntimeError as e:
    print("expected:", e)


🖥️   Card accepted. Please enter PIN.
🖥️   wrong PIN (1/3)
attempt 1: wrong PIN (1/3)
🖥️   wrong PIN (2/3)
attempt 2: wrong PIN (2/3)
🖥️   too many wrong PINs — card blocked
🖥️   Card ejected. Have a nice day!
attempt 3: too many wrong PINs — card blocked
card blocked? True
ATM state after block: State.IDLE
expected: card blocked — keeping card


In [7]:
# -- cannot withdraw without authenticating --------------------------------
carol = Customer("Carol", {AccountType.CHECKING: Account("C-CHK", AccountType.CHECKING, 400)})
carol_card = Card("CARD-3", "1111", carol)
bank.register(carol_card)

atm.insert_card(carol_card)
try:
    atm.run(Withdraw(carol.accounts[AccountType.CHECKING], 50))
except RuntimeError as e:
    print("expected:", e)
atm.eject()


🖥️   Card accepted. Please enter PIN.
expected: not authenticated
🖥️   Card ejected. Have a nice day!


In [8]:
# -- daily withdraw limit --------------------------------------------------
atm.insert_card(carol_card)
atm.enter_pin("1111")
chk_c = carol.accounts[AccountType.CHECKING]
chk_c.daily_withdraw_limit = 100  # small limit for demo

atm.run(Withdraw(chk_c, 80))
try:
    atm.run(Withdraw(chk_c, 50))  # 80 + 50 > 100
except RuntimeError as e:
    print("expected:", e)
atm.eject()


🖥️   Card accepted. Please enter PIN.
🖥️   Welcome, Carol!
🖥️   Please take $80.00
🧾 RECEIPT
    Withdraw from C-CHK
    Amount: $80.00
    New balance: $320.00
   -----
expected: daily limit $100 exceeded (already took $80)
🖥️   Card ejected. Have a nice day!


In [9]:
# -- ATM physically out of cash --------------------------------------------
small_atm = ATM(bank, CashDispenser(cash=20.0), DepositSlot(), Screen(), Printer(),
                today=date(2026, 4, 20))
small_atm.insert_card(card)
small_atm.enter_pin("1234")
try:
    small_atm.run(Withdraw(chk, 100))  # account has money, machine does not
except RuntimeError as e:
    print("expected:", e)
print("account balance unchanged:", chk.balance)  # must be untouched
small_atm.eject()


🖥️   Card accepted. Please enter PIN.
🖥️   Welcome, Alice!
expected: ATM cannot dispense — not enough cash in machine
account balance unchanged: 600.0
🖥️   Card ejected. Have a nice day!


## 7) Try it yourself — extensions

Good practice: try to implement one of these on top of the code above.

1. **Receipt toggle**: let the user choose "no receipt" — the `ATM` shouldn't print one.
2. **Mini-statement**: a new `MiniStatement(Transaction)` that prints the last 5 entries from `atm.log` for the card's accounts.
3. **PayBill**: add a `PayBill(Transaction)` that sends money to an external payee. Notice how you don't need to modify `ATM` at all — that's the Open/Closed Principle paying off.
4. **Concurrency**: two ATMs share the same `Bank`. What happens if both withdraw from the same account at once? Add a `threading.Lock` on `Account` and test.
5. **Timeout**: if the user takes longer than N seconds between `enter_pin` and `run`, auto-eject.

### What you practiced
- **Single Responsibility** — hardware, bank and transactions each live in their own class.
- **Command pattern** — `Transaction` + its subclasses.
- **State machine** — the `State` enum + guards in `ATM` methods.
- **Dependency injection** — the ATM receives its hardware and bank, so you can swap fakes in tests.
